# Notebook 4: BiLSTM with BLOSUM62

The confusion matrix from the XGBoost model shows that E (β-strand) errors are split
almost evenly between H and C — the model has no useful signal for strands and essentially
guesses. This is expected: strand pairing is determined by hydrogen bonds between residues
that can be far apart in sequence. A 15-residue window cannot see this.

A **Bidirectional LSTM** processes the entire protein sequence in both directions,
so every residue's prediction is informed by full sequence context — not just its neighbours.
This directly addresses the non-locality problem.

### Architecture
- Input: BLOSUM62 vectors (20-dim per residue)
- BiLSTM: 2 layers × 128 hidden units × 2 directions = 256-dim context per residue
- Classifier: Linear(256 → 3)
- Loss: CrossEntropy with `ignore_index=-1` to skip padded positions


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from Bio.Align import substitution_matrices
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df = pd.read_csv("protein_sequences_ss.csv")
print(f"Loaded {len(df)} proteins")


In [ ]:
# ── BLOSUM62 setup ────────────────────────────────────────────────────────────
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

blosum = substitution_matrices.load("BLOSUM62")
blosum_matrix = np.array([
    [float(blosum[(aa1, aa2)]) for aa2 in AMINO_ACIDS]
    for aa1 in AMINO_ACIDS
])

def blosum_encode(aa):
    if aa not in aa_to_index:
        return np.zeros(20)
    return blosum_matrix[aa_to_index[aa]]

# Label encoder
le = LabelEncoder()
le.fit(["H", "E", "C"])
print("Classes:", le.classes_)


In [ ]:
# ── Protein-level split ───────────────────────────────────────────────────────
protein_ids = df.index.tolist()
train_ids, test_ids = train_test_split(protein_ids, test_size=0.2, random_state=42)
train_df = df.iloc[train_ids].reset_index(drop=True)
test_df  = df.iloc[test_ids].reset_index(drop=True)
print(f"Train: {len(train_df)} proteins | Test: {len(test_df)} proteins")


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
# Each sample is a whole protein, not a window.
# x shape: (seq_len, 20)   y shape: (seq_len,)
class ProteinDataset(Dataset):
    def __init__(self, df):
        self.samples = []
        for _, row in df.iterrows():
            x = torch.tensor(
                np.array([blosum_encode(aa) for aa in row["sequence"]]),
                dtype=torch.float32
            )
            y = torch.tensor(
                le.transform(list(row["secondary_structure"])),
                dtype=torch.long
            )
            self.samples.append((x, y))

    def __len__(self):             return len(self.samples)
    def __getitem__(self, idx):    return self.samples[idx]


In [ ]:
# ── Collate: pad variable-length sequences to batch ──────────────────────────
def collate_fn(batch):
    xs, ys  = zip(*batch)
    lengths = [x.shape[0] for x in xs]
    max_len = max(lengths)

    x_pad = torch.zeros(len(xs), max_len, 20)
    y_pad = torch.full((len(xs), max_len), fill_value=-1, dtype=torch.long)

    for i, (x, y) in enumerate(zip(xs, ys)):
        x_pad[i, :x.shape[0]] = x
        y_pad[i, :y.shape[0]] = y

    return x_pad, y_pad, lengths


In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class BiLSTM(nn.Module):
    def __init__(self, input_size=20, hidden_size=128, num_layers=2,
                 num_classes=3, dropout=0.3):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        # hidden_size * 2: forward + backward hidden states concatenated
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.bilstm(x)       # (batch, seq_len, hidden*2)
        return self.classifier(out)   # (batch, seq_len, num_classes)


In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_dataset = ProteinDataset(train_df)
test_dataset  = ProteinDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)


In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
model     = BiLSTM().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-1)   # -1 = padded positions
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader):
    model.train()
    total = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x).view(-1, 3), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def compute_loss(model, loader):
    model.eval()
    total = 0
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            total += criterion(model(x).view(-1, 3), y.view(-1)).item()
    return total / len(loader)

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y, lengths in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x).argmax(dim=-1)
            for i, l in enumerate(lengths):
                preds.extend(out[i, :l].cpu().numpy())
                labels.extend(y[i,  :l].cpu().numpy())
    return np.array(preds), np.array(labels)


In [ ]:
# ── Training loop with early stopping ─────────────────────────────────────────
NUM_EPOCHS    = 40
PATIENCE      = 5    # stop if val loss doesn't improve for 5 checks
best_val_loss = float("inf")
patience_ctr  = 0

train_losses, val_losses, val_accs = [], [], []

for epoch in range(NUM_EPOCHS):
    tr_loss = train_epoch(model, train_loader)

    if (epoch + 1) % 5 == 0:
        vl_loss           = compute_loss(model, test_loader)
        preds, labels_arr = evaluate(model, test_loader)
        acc               = (preds == labels_arr).mean()

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        val_accs.append(acc)

        print(f"Epoch {epoch+1:02d} | Train Loss: {tr_loss:.4f} | "
              f"Val Loss: {vl_loss:.4f} | Accuracy: {acc:.4f}")

        # Early stopping
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), "bilstm_best.pt")
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}.")
                break

print("\nTraining complete. Best model saved to bilstm_best.pt")


In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
epochs_logged = list(range(5, 5 * len(train_losses) + 1, 5))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(epochs_logged, train_losses, label="Train loss", marker="o")
ax1.plot(epochs_logged, val_losses,   label="Val loss",   marker="o")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training vs Validation Loss")
ax1.legend()

ax2.plot(epochs_logged, val_accs, color="green", marker="o")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()


In [ ]:
# ── Final evaluation (best model) ─────────────────────────────────────────────
model.load_state_dict(torch.load("bilstm_best.pt", map_location=device))

preds, labels_arr = evaluate(model, test_loader)
print(classification_report(labels_arr, preds, target_names=le.classes_))
